<a href="https://colab.research.google.com/github/lahnabek/Kaggle-Challenge/blob/lahna/my_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment runner (DINOv2 + adapters)

This notebook drives the **histopathology OOD patch classification** pipeline.  
All implementation lives under the importable package **`utils/`** so `DataLoader(num_workers>0)` works with multiprocessing `spawn` (macOS, Colab).

**Quick start:** run the first code cell, then edit **`RUNS`** and execute the launch cell.


In [1]:
# !git clone https://github.com/lahnabek/Kaggle-Challenge.git
# %cd <repo>

In [2]:
# Hugging Face auth notes
# - Store your token in an env var (e.g. HF_TOKEN) or run `huggingface-cli login` once.
# - UNI / UNI2-h / Virchow2 are gated models, so you must have access approved.

# (Examples)
# import timm
# model = timm.create_model("hf-hub:MahmoodLab/uni", pretrained=True)
# model = timm.create_model("hf-hub:paige-ai/Virchow2", pretrained=True)
# model = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True)

- ajouter transfo imagenet
- tester avec diff resize en multiple pour voir l'effet (98 vs 112)
- tester fp16 vs fp32
- tester pour vircho CLS seul et CLS + mean patch

In [3]:
# import timm
# from timm.data import resolve_data_config
# from timm.data.transforms_factory import create_transform
# from huggingface_hub import login

# login()  # login with your User Access Token, found at https://huggingface.co/settings/tokens

# # pretrained=True needed to load UNI2-h weights (and download weights for the first time)
# timm_kwargs = {
#             'img_size': 224,
#             'patch_size': 14,
#             'depth': 24,
#             'num_heads': 24,
#             'init_values': 1e-5,
#             'embed_dim': 1536,
#             'mlp_ratio': 2.66667*2,
#             'num_classes': 0,
#             'no_embed_class': True,
#             'mlp_layer': timm.layers.SwiGLUPacked,
#             'act_layer': torch.nn.SiLU,
#             'reg_tokens': 8,
#             'dynamic_img_size': True
#         }
# model = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, **timm_kwargs)
# transform = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))
# model.eval()


# TO DO (research)

- Tissue / artefact QC (e.g. all-black or all-white patches)
- Stain augmentation and normalization (H&E literature)
- Alternative backbones: CTransPath, UNI, Virchow
- Adapters: AdaptFormer, LoRA / VeRA

## Notes augmentation (taille effective du dataset)

- **Batch augmentation / repeated augmentation**: répéter une même image dans le batch avec des augmentations différentes (utile pour tester l’impact de “augmenter la taille effective”). Référence: Hoffer et al., *Augment Your Batch: Improving Generalization Through Instance Repetition*, CVPR 2020.
- **Mixup / CutMix**: augmentations au niveau batch qui créent de nouveaux exemples (sans stocker un dataset augmenté).
- **AugMix / consistency**: plusieurs vues + régularisation de cohérence (plutôt orienté robustesse).


## 0) Environment and imports


Add the repository root to `sys.path` (required in Jupyter / Colab when the package is not installed with `pip`).

The first cell imports configuration types, `run_experiment`, and prints `DEVICE`.


In [ ]:
import os
import sys
from pathlib import Path

# Repository root (run the notebook from the repo root, or adjust this path in Colab)
_ROOT = Path.cwd().resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import pandas as pd

from utils.constants import DEVICE, RUNS_DIR, TEST_IMAGES_PATH, TRAIN_IMAGES_PATH, VAL_IMAGES_PATH
from utils.common import ensure_dir, safe_json, seed_everything
from utils.config import (
    AugmentationConfig,
    EarlyStoppingConfig,
    ModelConfig,
    ModuleSpec,
    ProcessingConfig,
    RunConfig,
    TrainConfig,
)
from utils.model import DefaultBinaryHead
from utils.outliers import MethodCOutlierParams
from utils.experiment import run_experiment
from utils.data_augmentation import make_train_augmentations

# Hugging Face auth (needed for gated models like UNI / UNI2-h / Virchow2).
# If you already ran `huggingface-cli login`, this is optional.

from huggingface_hub import login

_hf_token = ""
if _hf_token:
    login(token=_hf_token)


print(f"Using device: {DEVICE}")
print(f"H5 paths (override in utils/constants.py if needed): train={TRAIN_IMAGES_PATH!r}")


Using device: cpu
H5 paths (override in utils/constants.py if needed): train='train.h5'


### Method C outlier detection (`ExtractOutlier`)

Implementation: **`utils/outliers.py`** (same rules as `outlier_detection.ipynb` on **raw H5** resolution).

- **Train:** flagged patch IDs are **removed** from training.
- **Val / test:** outliers stay in the loader; predictions are **0** without a forward pass.

Parallelism for the scan uses **threads** (`ThreadPoolExecutor`), controlled by `outlier_scan_workers` in `RunConfig`.


## 1) Experiment configuration


Edit the **`RUNS`** list in the last section. Types are dataclasses in **`utils/config.py`**:

- `ProcessingConfig` — resize and optional numpy / sklearn steps  
- `ModelConfig` — DINO backbone name, `ModuleSpec` for adapter and head  
- `TrainConfig` — batch size, LR, `num_workers` (PyTorch DataLoader), early stopping  
- `RunConfig` — run name, seeds, optional `MethodCOutlierParams`, `data_fraction` smoke tests, test prediction  

`ModuleSpec(enabled=False)` for the adapter triggers **one** frozen-backbone pass and **head-only** training on embeddings (linear probing).


(No code cell here — imports are in section 0.)


## 2) Data pipeline


**Module:** `utils/data.py`

- `H5BinaryDataset` — HDF5 patches, picklable for multiprocessing  
- `PreprocessingTransform` / `build_preprocessing` — picklable transforms  
- `EmbeddingTensorDataset` / `EmbeddingValDataset` — precomputed DINO features  
- `CenterProportionalBatchSampler` — optional center-balanced batches


## 3) Model


**Module:** `utils/model.py`

- `load_frozen_backbone` — DINOv2 from `torch.hub`  
- `DefaultMLPAdapter`, `DefaultBinaryHead`  
- `FullModel` — backbone + optional adapter + head  
- `HeadOnly` — linear probe on fixed embeddings


## 4) Training loop and metrics


**Module:** `utils/training.py`

`Trainer` performs train/val epochs, per-center metrics, CSV logging (`metrics.csv`), ROC/PR curve `.npz` files, early stopping, `best.pt` / `last.pt`.


## 5) Test prediction


**Module:** `utils/predict.py` — `predict_test(...)` writes `predictions.csv` when `do_predict_test=True` in `RunConfig`.


## 6) Experiment runner


**Module:** `utils/experiment.py` — `run_experiment(run_cfg)` orchestrates outlier filtering, DINO precompute (if adapter disabled), optimization, and optional test inference.

Outputs under `runs/<run_name>/seed_<k>/`: `config.json`, `metrics.csv`, `checkpoints/`, `curves/`.


## 7) Define `RUNS` and launch

Two baselines (toggle comments to switch):

1. **Baseline** — no outlier filter, full data.  
2. **Baseline + Method C** — same model, with `MethodCOutlierParams` and optional `data_fraction` for quick tests.

Both use **linear probing** (`adapter.enabled=False`): one backbone precompute, then head-only epochs.


### Summary of the two runs below

| Run | Outliers | Notes |
|-----|----------|--------|
| `baseline_no_outlier_filter` | Off | Full train/val |
| `baseline_method_c_outliers` | Method C | Same hyperparameters + outlier scan |

Both entries run sequentially; remove or comment any run you do not need.


AJOUTER VAL PREDICTION POUR CHECKER LES OUTLIRS

In [5]:
# Shared linear-probing setup: frozen backbone, single precompute, train head only.
# Toggle these two knobs per experiment.
USE_IMAGENET_NORM = True
USE_FP16 = False  # set True to enable AMP (fp16 autocast) during embedding precompute on CUDA

_DINO_MODEL = ModelConfig(
    backbone_name="dinov2_vits14",  # e.g. "uni", "uni2_h", "virchow2_cls", "virchow2_clsmean"
    use_fp16=USE_FP16,
    adapter=ModuleSpec(enabled=False, module_cls=None, module_kwargs=None),
    head=ModuleSpec(
        enabled=True,
        module_cls=DefaultBinaryHead,
        module_kwargs={},
    ),
)

_UNI_MODEL = ModelConfig(
    backbone_name="uni",  # e.g. "uni", "uni2_h", "virchow2_cls", "virchow2_clsmean"
    use_fp16=USE_FP16,
    adapter=ModuleSpec(enabled=False, module_cls=None, module_kwargs=None),
    head=ModuleSpec(
        enabled=True,
        module_cls=DefaultBinaryHead,
        module_kwargs={},
    ),
)

_UNI2_H_MODEL = ModelConfig(
    backbone_name="uni2_h",  # e.g. "uni", "uni2_h", "virchow2_cls", "virchow2_clsmean"
    use_fp16=USE_FP16,
    adapter=ModuleSpec(enabled=False, module_cls=None, module_kwargs=None),
    head=ModuleSpec(
        enabled=True,
        module_cls=DefaultBinaryHead,
        module_kwargs={},
    ),
)

_VIRCHOW2_CLS_MODEL = ModelConfig(
    backbone_name="virchow2_cls",  # e.g. "uni", "uni2_h", "virchow2_cls", "virchow2_clsmean"
    use_fp16=USE_FP16,
    adapter=ModuleSpec(enabled=False, module_cls=None, module_kwargs=None),
    head=ModuleSpec(
        enabled=True,
        module_cls=DefaultBinaryHead,
        module_kwargs={},
    ),
)

_VIRCHOW2_CLSMEAN_MODEL = ModelConfig(
    backbone_name="virchow2_clsmean",  # e.g. "uni", "uni2_h", "virchow2_cls", "virchow2_clsmean"
    use_fp16=USE_FP16,
    adapter=ModuleSpec(enabled=False, module_cls=None, module_kwargs=None),
    head=ModuleSpec(
        enabled=True,
        module_cls=DefaultBinaryHead,
        module_kwargs={},
    ),
)




In [6]:
# Data augmentation (train-only). Parameters match Faryna et al. defaults (n=3, m=5).
TRAIN_AUG = make_train_augmentations(
    use_he_randaugment=True,
    rand_n=3,
    rand_m=5,
    use_color_jitter=True,
    jitter_brightness=0.1,
    jitter_contrast=0.1,
    jitter_saturation=0.1,
)

# Observed (empirical) center proportions in train (example numbers).
center_proportions = {
    0: 17756 / 100_000,
    3: 38756 / 100_000,
    4: 43488 / 100_000,
}

# Option 1: force uniform per-center exposure (oversample minor centers with replacement).
_UNIFORM_CENTER_TRAIN = TrainConfig(
    num_workers=8,
    batch_size=16,
    lr=1e-3,
    num_epochs=100,
    early_stopping=EarlyStoppingConfig(monitor="val_loss", mode="min", patience=10),
    use_center_balanced_batches=True,
    center_proportions=None,
    center_sampling="uniform",
)

# Option 2: keep empirical proportions but still build center-proportional batches.
_BALANCED_TRAIN = TrainConfig(
    num_workers=8,
    batch_size=16,
    lr=1e-3,
    num_epochs=100,
    early_stopping=EarlyStoppingConfig(monitor="val_loss", mode="min", patience=10),
    use_center_balanced_batches=True,
    center_proportions=center_proportions,
    center_sampling="empirical",
)

_BASELINE_TRAIN = TrainConfig(
    num_workers=8,
    batch_size=16,
    lr=1e-3,
    num_epochs=100,
    early_stopping=EarlyStoppingConfig(monitor="val_loss", mode="min", patience=10),
    use_center_balanced_batches=False,
)


In [7]:
print(abc)

NameError: name 'abc' is not defined

In [ ]:
RUNS: list[RunConfig] = [
    # Baseline (no augmentation)
    RunConfig(
        run_name="baseline_norm",
        seeds=[0],
        processing=ProcessingConfig(resize_hw=(98, 98), imagenet_normalize=True),
        model=_DINO_MODEL,
        train=_BASELINE_TRAIN,
        outlier_params=None,
        data_fraction=None,
        do_predict_test=True,
    ),
    # Train-only stain augmentation (Faryna et al. defaults) + uniform center exposure.
    # Multi-view option: include_original + k_aug_views implements "batch augmentation" (Augment Your Batch).
    RunConfig(
        run_name="he_randaugment_uniform",
        seeds=[0],
        processing=ProcessingConfig(
            resize_hw=(98, 98),
            imagenet_normalize=True,
            extra_transform=TRAIN_AUG,
            augmentation=AugmentationConfig(include_original=True, k_aug_views=1),
        ),
        model=_DINO_MODEL,
        train=_UNIFORM_CENTER_TRAIN,
        outlier_params=None,
        data_fraction=None,
        do_predict_test=True,
    ),
]

results = []
for rc in RUNS:
    print(f"\n=== Starting run: {rc.run_name} ===")
    out = run_experiment(rc)
    results.append(out)

rows = []
for r in results:
    for sr in r["seed_results"]:
        rows.append(
            {
                "run_name": r["run_name"],
                "seed": sr["seed"],
                "best_epoch": sr.get("best_epoch"),
                "monitor": sr.get("early_stopping_monitor"),
                "best_monitor_value": sr.get("best_monitor_value"),
                "time_sec": sr.get("time_sec"),
            }
        )

pd.DataFrame(rows).sort_values(["run_name", "seed"])


=== Starting run: baseline_norm ===


Using cache found in /Users/user/.cache/torch/hub/facebookresearch_dinov2_main
/Users/user/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/user/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/user/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


precompute train seed=0:   0%|          | 0/6250 [00:27<?, ?it/s]

precompute val seed=0:   0%|          | 0/2182 [00:26<?, ?it/s]

Using cache found in /Users/user/.cache/torch/hub/facebookresearch_dinov2_main


...

train:   0%|          | 0/6250 [00:00<?, ?it/s]

eval:   0%|          | 0/2182 [00:00<?, ?it/s]

/Users/user/Desktop/GitHub/mva-dlmi-2026-histopathology-ood-classification/utils/experiment.py:348: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(os.path.j

predict:   0%|          | 0/85054 [00:26<?, ?it/s]

,run_name,seed,best_epoch,monitor,best_monitor_value,time_sec
0,baseline_norm,0,10,val_loss,0.295499,201.63953


In [ ]:
print(abc)

In [ ]:

# ---- RUNS ----
# Keep your examples below as comments, but generate a full batch of backbones.

DATA_FRAC = 0.01
SEEDS = [0]

# Resize as close as possible to original 96x96 patches:
# - UNI (ViT-L/16) supports multiples of 16 -> 96x96 is valid.
# - DINOv2 / UNI2-h / Virchow2 are patch14-based -> use 98x98 (7*14).
RESIZE_BY_BACKBONE: dict[str, tuple[int, int]] = {
    "dinov2_vits14": (98, 98),
    "uni": (96, 96),
    "uni2_h": (98, 98),
    "virchow2_cls": (98, 98),
    "virchow2_clsmean": (98, 98),
}

BACKBONES = [

    "uni",
    "uni2_h",
    "virchow2_cls",
    "virchow2_clsmean",
    "dinov2_vits14",
]

def make_model_cfg(backbone_name: str) -> ModelConfig:
    return ModelConfig(
        backbone_name=backbone_name,
        use_fp16=USE_FP16,
        adapter=ModuleSpec(enabled=False, module_cls=None, module_kwargs=None),
        head=ModuleSpec(enabled=True, module_cls=DefaultBinaryHead, module_kwargs={}),
    )

# 1) Baseline: no outlier scan (example)
# RunConfig(
#     run_name="baseline",
#     seeds=[0],
#     processing=ProcessingConfig(resize_hw=(98, 98), imagenet_normalize=USE_IMAGENET_NORM),
#     model=make_model_cfg("dinov2_vits14"),
#     train=_BASELINE_TRAIN,
#     outlier_params=None,
#     data_fraction=None,
#     do_predict_test=True,
# ),

# 2) Baseline + Method C outlier filter (example)
# RunConfig(
#     run_name="baseline_outliers6",
#     seeds=[0],
#     processing=ProcessingConfig(resize_hw=(98, 98), imagenet_normalize=USE_IMAGENET_NORM),
#     model=make_model_cfg("dinov2_vits14"),
#     train=_BASELINE_TRAIN,
#     data_fraction=None,
#     outlier_params=MethodCOutlierParams(
#         min_sat_mean=0.03,
#         min_colorfulness=0.03,
#         min_grad_energy=0.005,
#         white_gray_min=0.86,
#         white_max_sat=0.28,
#         max_white_frac=0.6,
#     ),
#     outlier_scan_workers=8,
#     outlier_scan_batch_size=1000,
#     do_predict_test=True,
# ),

# Batch: linear probing across backbones
RUNS: list[RunConfig] = []
for bb in BACKBONES:
    resize_hw = RESIZE_BY_BACKBONE[bb]
    run_name = (
        f"{bb}_r{resize_hw[0]}_norm"
    )
    RUNS.append(
        RunConfig(
            run_name=run_name,
            seeds=SEEDS,
            processing=ProcessingConfig(
                resize_hw=resize_hw,
                imagenet_normalize=USE_IMAGENET_NORM,
                extra_transform=TRAIN_AUG,
            ),
            model=make_model_cfg(bb),
            train=_UNIFORM_CENTER_TRAIN,
            outlier_params=None,
            data_fraction=DATA_FRAC,
            do_predict_test=True,
        )
    )

results = []
for rc in RUNS:
    print(f"\n=== Starting run: {rc.run_name} ===")
    out = run_experiment(rc)
    results.append(out)

rows = []
for r in results:
    for sr in r["seed_results"]:
        rows.append(
            {
                "run_name": r["run_name"],
                "seed": sr["seed"],
                "best_epoch": sr.get("best_epoch"),
                "monitor": sr.get("early_stopping_monitor"),
                "best_monitor_value": sr.get("best_monitor_value"),
                "time_sec": sr.get("time_sec"),
            }
        )

pd.DataFrame(rows).sort_values(["run_name", "seed"])


In [ ]:
# ---- Local-only test prediction from existing runs ----
# Useful when Colab runs out of memory for test inference.

import json
from pathlib import Path

import torch

from utils.config import ModelConfig, ModuleSpec, ProcessingConfig
from utils.data import build_preprocessing
from utils.model import DefaultBinaryHead, FullModel, load_frozen_backbone
from utils.predict import predict_test

# Put the run folder names you want to export predictions for.
RUN_NAMES = [
    "uni_r96_norm",
    "uni2_h_r98_norm",
    "virchow2_cls_r98_norm",
    "virchow2_clsmean_r98_norm",
]

# Which seeds to export (most runs use seed_0).
SEEDS_TO_EXPORT = [0]

for run_name in RUN_NAMES:
    cfg_path = Path(RUNS_DIR) / run_name / "config.json"
    if not cfg_path.exists():
        print(f"[skip] missing config: {cfg_path}")
        continue

    cfg = json.load(open(cfg_path))
    processing = cfg["processing"]
    model_cfg = cfg["model"]
    train_cfg = cfg["train"]
    threshold = float(cfg.get("predict_threshold", 0.5))

    # Rebuild the same eval preprocessing (no augmentation at test time).
    transform_eval = build_preprocessing(
        ProcessingConfig(
            resize_hw=tuple(processing["resize_hw"]),
            imagenet_normalize=bool(processing["imagenet_normalize"]),
            extra_transform=None,
        )
    )

    for seed in SEEDS_TO_EXPORT:
        run_dir = Path(RUNS_DIR) / run_name / f"seed_{seed}"
        ckpt_path = run_dir / "checkpoints" / "best.pt"
        if not ckpt_path.exists():
            print(f"[skip] missing checkpoint: {ckpt_path}")
            continue

        ckpt = torch.load(str(ckpt_path), map_location="cpu")

        if ckpt.get("head_only"):
            # Linear probe checkpoint: load frozen backbone + head weights only.
            bb = load_frozen_backbone(ckpt["backbone_name"])
            infer_cfg = ModelConfig(
                backbone_name=str(ckpt["backbone_name"]),
                adapter=ModuleSpec(enabled=False, module_cls=None, module_kwargs=None),
                head=ModuleSpec(enabled=True, module_cls=DefaultBinaryHead, module_kwargs={}),
            )
            infer_model = FullModel(bb, infer_cfg).to(DEVICE)
            sd = ckpt["model_state"]
            head_sd = {k.replace("head.", "", 1): v for k, v in sd.items() if k.startswith("head.")}
            infer_model.head.load_state_dict(head_sd)
        else:
            # FullModel checkpoint: rebuild backbone from config and load full state.
            bb = load_frozen_backbone(model_cfg["backbone_name"])
            infer_cfg = ModelConfig(
                backbone_name=model_cfg["backbone_name"],
                adapter=ModuleSpec(enabled=bool(model_cfg["adapter"]["enabled"]), module_cls=None, module_kwargs=None),
                head=ModuleSpec(enabled=True, module_cls=DefaultBinaryHead, module_kwargs={}),
            )
            infer_model = FullModel(bb, infer_cfg).to(DEVICE)
            infer_model.load_state_dict(ckpt["model_state"])

        print(f"\n=== Export predictions: {run_name} seed={seed} ===")
        predict_test(
            run_dir=str(run_dir),
            model=infer_model,
            transform=transform_eval,
            threshold=threshold,
            test_outlier_ids=None,
            subset_ids=None,
            num_workers=4,
        )
        print(f"wrote: {run_dir / 'predictions.csv'}")


/var/folders/t0/gw753q351rbc1y6tksb52wm00000gn/T/ipykernel_29815/2565511753.py:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(str(ckpt_path), map_locati


=== Export predictions: uni_r96_norm seed=0 ===


predict:   0%|          | 0/85054 [00:15<?, ?it/s]